In [0]:
from pyspark.sql.functions import col, to_timestamp

print("Initializing Silver Layer Processing...")

# 1. Define the Bronze path (Your Unity Catalog Volume)
bronze_path = "/Volumes/project_streame/bronze_layer/batches/"

# 2. Read the raw JSON data into a DataFrame
# Spark is smart enough to read all JSON files in this directory at once
raw_df = spark.read.json(bronze_path)

print("1. Raw Bronze Schema (Notice the nested struct types):")
raw_df.printSchema()

# 3. The Flattening Logic
# We use '.*' to extract every sub-field from a nested struct column
flattened_df = raw_df.select(
    col("event_id"),
    col("event_timestamp"),
    col("event_details.*"), # Expands into 'action' and 'device'
    col("user_data.*"),     # Expands into 'user_id'
    col("transaction.*") 
)

# 4. Data Quality Enforcement (Type Casting)
# Real-world data needs strict types.so for that purpose We cast the string timestamp into a true TimestampType.
silver_df = flattened_df.withColumn(
    "event_timestamp", 
    to_timestamp(col("event_timestamp"))
)

# Defensive programming: Drop any exact duplicate rows that might have snuck in
silver_df = silver_df.dropDuplicates(["event_id"])

print("\n2. Cleaned Silver Schema (Completely flat and strongly typed):")
silver_df.printSchema()

# 5. Displaying the final result
display(silver_df)

In [0]:
print("Writing Silver Layer to Unity Catalog...")

# 1. Defining the Silver path inside  Unity Catalog Volume
# created a new folder specifically for the cleaned data
silver_path = "/Volumes/project_streame/silver_layer/cleaned_data/"

# 2. Writing the DataFrame as a Parquet file
# mode("overwrite") ensuring that if we rerun this notebook, it replaces the old data 
# instead of duplicating it.
silver_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(silver_path)

print(f"Success! Cleaned Parquet files saved to {silver_path}")

# 3. Verifyimg the files exist
display(dbutils.fs.ls(silver_path))